# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2 clinical dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

This dataset contains detailed clinicopathological and molecular characteristics from 77 cancer survivors with second primary colorectal cancer, including demographics, comorbidities, diagnosis intervals, anatomical and molecular data, and more.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL to the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f'Dataset loaded: {getattr(metadata, "name", "(no name)")}')
if hasattr(metadata, 'description'):
    print(metadata.description)

## 2. Data Overview
List available record sets within the dataset, as well as their fields and corresponding `@id` identifiers.

This helps to discover what data structures and fields are available for further analysis. All accesses by `@id`.

In [ ]:
# Discover all record sets and their fields
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record_sets:")
for rs in record_sets:
    print(f"- RecordSet: {getattr(rs, '@id', '-')}")
    if hasattr(rs, 'fields'):
        print("    Fields:")
        for field in rs.fields:
            print(f"      - {getattr(field, '@id', '-')}: {getattr(field, 'name', '-')}")

## 3. Data Extraction
Load records from each record set into DataFrames for analysis.

For this example, we'll extract every record set discovered above. All names/fields are referred to by their `@id`.

In [ ]:
# Prepare to load all record sets as DataFrames, referenced by their @id
dfs = {}

for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    if not rs_id:
        continue
    # Load records as list of dicts
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        if not df.empty:
            dfs[rs_id] = df
            print(f'RecordSet @id: {rs_id} - Loaded {len(df)} records. Columns: {list(df.columns)}')
        else:
            print(f'RecordSet @id: {rs_id} - No records found.')
    except Exception as e:
        print(f'Could not load records for @id: {rs_id}: {e}')

### View columns and a data sample from the main record set

Below, we'll display the columns of the primary record set (the largest non-empty DataFrame detected) and preview its data. If you know the clinical table's exact `@id`, you can reference it directly. Otherwise, we use the first large DataFrame.

In [ ]:
# Identify a primary tabular record set for demonstration
main_rs_id = None
maxlen = 0
for rs_id, df in dfs.items():
    if len(df) > maxlen:
        maxlen = len(df)
        main_rs_id = rs_id

if main_rs_id:
    print(f'Chosen RecordSet @id for demo: {main_rs_id}')
    print('Columns:', dfs[main_rs_id].columns.tolist())
    display(dfs[main_rs_id].head())
else:
    print('No main record set found!')

## 4. Exploratory Data Analysis (EDA)
Process the main clinical table: filtering, normalization, and simple grouping by categorical columns.

All column references are by their full `@id` as obtained from the Data Overview above.

In [ ]:
import numpy as np

# List available numeric fields for possible EDA
df = dfs[main_rs_id]
possible_numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f'Numeric columns in {main_rs_id}: {possible_numeric_cols}')

# If a numeric column exists, pick the first (typically e.g. Age or interval_days)
if possible_numeric_cols:
    numeric_field_id = possible_numeric_cols[0]
    print(f'Using numeric field: {numeric_field_id}')
    threshold = np.nanpercentile(df[numeric_field_id], 50)  # Use median as example filter
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)})")
    # Normalization (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Find a categorical field to group by
    possible_cat_cols = [col for col in df.columns if df[col].dtype == object and len(df[col].unique()) < len(df) // 2]
    if possible_cat_cols:
        group_col_id = possible_cat_cols[0]
        print(f'Grouping by: {group_col_id}')
        grouped_df = filtered_df.groupby(group_col_id)[numeric_field_id].agg(['mean','count'])
        display(grouped_df.head())
else:
    print('No numeric fields found for EDA in this record set.')

## 5. Visualization
Let's visualize the distribution of our selected numeric field (e.g., Age or interval) and compare its distribution by a group attribute, if found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

if possible_numeric_cols:
    fig, axs = plt.subplots(1, 2, figsize=(12,4))
    # Histogram
    sns.histplot(df[numeric_field_id], kde=True, ax=axs[0], bins=10)
    axs[0].set_title(f'Distribution of {numeric_field_id}')
    axs[0].set_xlabel(numeric_field_id)
    # Boxplot by group if available
    if possible_cat_cols:
        sns.boxplot(data=df, x=group_col_id, y=numeric_field_id, ax=axs[1])
        axs[1].set_title(f'{numeric_field_id} by {group_col_id}')
        axs[1].set_xlabel(group_col_id)
        axs[1].set_ylabel(numeric_field_id)
    else:
        df[numeric_field_id].plot.box(ax=axs[1])
        axs[1].set_title(f'Boxplot of {numeric_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We loaded and explored the FAIR² clinicopathological dataset using its Croissant metadata and the `mlcroissant` library.

- All dataset elements (record sets, fields, columns) were referenced by their `@id`.
- We demonstrated dynamic extraction, filtering, normalization, grouping, and visualization of clinical fields.
- The Croissant schema and `mlcroissant` allow clear, reproducible data wrangling and facilitate compliant FAIR data science.

For further work, see field documentation in the Croissant JSON-LD for full variable meaning, and consider exporting EDA results or cleaning scripts for future analytic pipelines.